In [2]:
import Defaults
from qiskit.circuit.library import QFTGate
from qiskit import QuantumRegister, ClassicalRegister, QuantumCircuit
from math import gcd, ceil, log2
from fractions import Fraction
import numpy as np
import matplotlib.pyplot as plt
from time import time
from Gates import AdditionalGates as QC

t=[]

plt.rcParams["figure.figsize"] = (20,5)
style = "latex"

from IPython.core.magic import register_cell_magic
from IPython import get_ipython
from IPython.display import display

@register_cell_magic
def skip(line, cell):
    return

@register_cell_magic
def skip_if(line, cell):
    if eval(line):
        return
    get_ipython().run_cell(cell)

In [9]:
#This box constructs the quantum circuit for the 2 numbers at the top

a=2
N=15

if gcd(a, N) > 1: #Checks that "N" isn't already factorisable by "a"
    print(f"Error: gcd({a},{N}) > 1")
    quit()

k=2*ceil(log2(N)) #number of multiplication gates required
b_list = [pow(a,2**val,N) for val in range(k)] #get the value that each multiplication gate multiplies by

work_bits = k//2 #stores workings
phase_bits = k # stores power of 2 we are multiplying the work register by

Phase_Register = QuantumRegister(phase_bits,"P")
Work_Register = QuantumRegister(work_bits,"W")
Carry = QuantumRegister(work_bits+1,"C")
Accumulator1 = QuantumRegister(work_bits,"A1")
Flag = QuantumRegister(1,"F")
Ancilla = QuantumRegister(1,"anc")
meas = ClassicalRegister(phase_bits,"meas")
qc = QC(Phase_Register, Work_Register, Carry, Accumulator1, Flag, Ancilla, meas)
qc.h(Phase_Register)
qc.x(Work_Register[0])

for index in range(len(b_list)):
    b = b_list[index] #using the value we determined earlier...
    if b > 1: #...as long as we are not multiplying by 1...
        qc.const_mult_mod_self(Work_Register, b, N, Carry, Accumulator1, Flag, Phase_Register[index], Ancilla) #...We apply the multiplication here

qc.compose(QFTGate(phase_bits).inverse(),Phase_Register,inplace=True)
qc.measure(Phase_Register, meas)

qc.count_ops()

OrderedDict([('ccx', 8550),
             ('cx', 4915),
             ('x', 801),
             ('h', 10),
             ('measure', 10),
             ('qft_dg', 1)])

In [7]:
sim = Defaults.Simulation()

print("Qubits:",sim.backend.num_qubits)

Qubits: 31


In [10]:
#This box finds the smallest amount of gates possible to use that the compiler natively supports

t.append(time())
decomp = [qc]
t.append(time())
gates = decomp[-1].count_ops()
print(f"decomposition 0: {str(float(0))[:6]}s, gate types: {str(len(gates)).zfill(3)}, number of gates: {str(sum(gates.values())).zfill(7)}, Gates: {list(gates.keys())}")
for i in range(0,10):
    t.append(time())
    decomp.append((decomp[-1] if len(decomp) > 0 else qc).decompose())
    t.append(time())
    new_gates = decomp[-1].count_ops()
    if new_gates == gates:
        del(decomp[-1])
        break
    else:
        gates = new_gates
    print(f"decomposition {i+1}: {str(t[-1]-t[-2])[:6]}s, gate types: {str(len(gates)).zfill(3)}, number of gates: {str(sum(gates.values())).zfill(7)}, Gates: {list(gates.keys())}")
smallest = -1
for i, circ in enumerate(decomp):
    t.append(time())
    sim.update_isa_circuit(circ)
    t.append(time())
    gates = sim.isa_circuit.count_ops()
    if smallest == -1 or sum(gates.values()) < sum(decomp[smallest].count_ops().values()) :
        smallest = i
    print(f"decomposition {i}: {str(t[-1]-t[-2])[:6]}s, gate types: {str(len(gates)).zfill(3)}, number of gates: {str(sum(gates.values())).zfill(7)}, Gates: {list(gates.keys())}")
sim.update_isa_circuit(decomp[smallest])

decomposition 0: 0.0s, gate types: 006, number of gates: 0014287, Gates: ['ccx', 'cx', 'x', 'h', 'measure', 'qft_dg']
decomposition 1: 0.2056s, gate types: 008, number of gates: 0134046, Gates: ['cx', 't', 'tdg', 'h', 'u', 'cp', 'measure', 'swap']
decomposition 2: 1.0828s, gate types: 004, number of gates: 0134236, Gates: ['p', 'cx', 'u', 'measure']
decomposition 3: 1.6118s, gate types: 003, number of gates: 0134236, Gates: ['u', 'cx', 'measure']
decomposition 0: 0.1346s, gate types: 006, number of gates: 0013021, Gates: ['ccx', 'cx', 'x', 'cp', 'h', 'measure']
decomposition 1: 1.7293s, gate types: 010, number of gates: 0110818, Gates: ['cx', 'tdg', 't', 'u2', 'h', 'p', 'unitary', 'u', 'cp', 'measure']
decomposition 2: 1.7018s, gate types: 007, number of gates: 0110968, Gates: ['cx', 'p', 'u2', 'u', 'u1', 'unitary', 'measure']
decomposition 3: 1.4364s, gate types: 006, number of gates: 0116176, Gates: ['u', 'cx', 'u2', 'u1', 'unitary', 'measure']


In [ ]:
#Starts the simulation job, and can be ran again to check if the job is complete

job_name = f"Shor({a},{N}) gates sample"
result = sim.get_result(job_name)
if result:
    if result == True:
        print("Waiting on result")
    else:
        print("Gathering Results")
        sim_probs = Defaults.get_probs(result,Defaults.generate_dict(phase_bits), percentage=True)
        graph_probs = Defaults.tidy_probs(sim_probs,0.01)
        Defaults.create_bar(sim_probs.keys(), sim_probs.values(), xlabel="Measurement outcomes", ylabel="Probability", title="Probability of Bell state").show()
        Defaults.create_bar(graph_probs.keys(), graph_probs.values(), xlabel="Measurement outcomes", ylabel="Probability", title="Probability of Bell state").show()
else:
    if result == False:
        print("Job was cancelled or errored")
    else:
        #sim.update_isa_circuit(qc)
        sim.save_job(sim.sampler.run([sim.isa_circuit],shots=1024),job_name)
        print("Job Started")

Job Started


In [9]:
#Once the simulation is complete, we can use the results to find a factor of N

%%skip_if result in [None, False, True]
counts = result[0].data.meas.get_counts()
counts_keep = {}
threshold = np.max(list(counts.values())) / 5

for key, value in counts.items():
    if value > threshold:
        counts_keep[key] = value

found = False
attempt = 0
while not found:
    bitstring = list(counts_keep.keys())[attempt]
    attempt += 1
    decimal = int(bitstring, 2)
    if decimal != 0:
        phase = decimal / (2**phase_bits)
        frac = Fraction(phase).limit_denominator(N)
        r = frac.denominator
        if r % 2 == 0:
            x = pow(a, r // 2, N) - 1
            d = gcd(x, N)
            print(r,x,d)
            if d > 1:
                found = True
                print(f"*** Non-trivial factor found: {x}, GCD({x},{N}): {gcd(x,N)} ***")

2 1 1
4 3 3
*** Non-trivial factor found: 3, GCD(3,15): 3 ***
